# POSE — Generalized 6D Pose Pipeline

**Wirf ein beliebiges CAD-Teil vorne rein → hinten kommt die rekonstruierte 3D-Szene raus.**

Dieses Notebook schaltet die Module der Pipeline modular aneinander. Jede Stage = eine Funktion, jede Funktion hat eine Visualisierung. Du kannst oben die Teile-Liste ändern und alles läuft generisch durch.

```
CAD (USD)                                                                 3D-Anlage
   │                                                                          ▲
   ▼                                                                          │
[1] Daten-Gen (Isaac SDG)  →  [2] Face-Atlas/Registry  →  [3] Training       │
     top-down RGB+depth+OBB        Faces pro Teil (R_face)     OBB-Det + Face-CNN
   │                                                                          │
   ▼                                                                          │
[5] Inferenz: Bild → Teile finden → OBB → Crop → klassifizieren → Face       │
   │                                                                          │
   ▼                                                                          │
[6] Alignment: Face→R_face · Yaw via Mask-MSE · (x,y)-Backprojection ────────┘
        → pose_result.json (Contract)  →  [7] 3D-Render (Nullpunkt, Klick-Info)
```

> Status: Module **[2] Face-Atlas, [5/6] Inferenz+Alignment, [7] 3D-Viewer** stehen und laufen mit Dummy/Fallback. **[3] Training** (OBB-Detektor + Face-CNN) ist *vorbereitet, nicht trainiert* — `infer()` nutzt solange einen nearest-template-Fallback. Roadmap: `gantt status`.


## 0 · Setup & Config
Eine Stelle für alle Pfade + die Teile-Liste. Ändere `PARTS`, der Rest ist generisch.

In [ ]:
import os, sys, json, subprocess, time
from pathlib import Path

REPO = Path("/Users/Admin/POSE")
sys.path.insert(0, str(REPO))

# GPU-Workstation (Isaac SDG + faces-venv). WoL falls aus.
GPU_HOST   = "max@100.85.216.95"
WOL_HOST   = "admin@100.117.146.46"
WOL_MAC    = "24:4b:fe:4b:79:e0"
BOX_REPO   = "/mnt/data/kip_pose"
BOX_ISAAC  = "/mnt/data/isaacsim-venv/bin/python"
BOX_FACES  = "/mnt/data/faces-venv/bin/python"

# >>> Hier Teile reinwerfen (USD-Dateinamen unter data/SDG/IsaacSim/USD-Files/) <<<
PARTS = ["Anker_Lang", "Zahnrad_Typ7", "Poltopf_kurz_centered",
         "Getriebegehaeuse_typ4", "Buerstenhalter_2polig"]

USD_DIR  = "data/SDG/IsaacSim/USD-Files"
REGISTRY = REPO / "registry"          # persistente Face-Registries pro Teil
OUT      = REPO / "data" / "output"
print("Repo:", REPO, "| Parts:", PARTS)

### Helpers — GPU-Box ansteuern (generisch, kein manuelles SSH)

In [ ]:
def box_up(timeout=3):
    return subprocess.run(["ssh","-o",f"ConnectTimeout={timeout}","-o","BatchMode=yes",
                           GPU_HOST,"true"], capture_output=True).returncode == 0

def wake_box(wait=60):
    if box_up(): return True
    subprocess.run(["ssh","-o","ConnectTimeout=8",WOL_HOST,f"wakeonlan {WOL_MAC}"], capture_output=True)
    for _ in range(wait//5):
        time.sleep(5)
        if box_up(): return True
    return False

def on_box(cmd):
    "Run a shell command on the GPU box, return stdout."
    r = subprocess.run(["ssh","-o","ConnectTimeout=8",GPU_HOST,cmd], capture_output=True, text=True)
    if r.returncode: print("[box stderr]", r.stderr[-800:])
    return r.stdout

def push(localrel):  # rsync a repo subdir up to the box
    subprocess.run(["rsync","-az",str(REPO/localrel)+"/",
                    f"{GPU_HOST}:{BOX_REPO}/{localrel}/"], check=False)

def pull(remoterel, localrel):  # rsync a box subdir down
    (REPO/localrel).mkdir(parents=True, exist_ok=True)
    subprocess.run(["rsync","-az",f"{GPU_HOST}:{BOX_REPO}/{remoterel}/",str(REPO/localrel)+"/"], check=False)

print("GPU box reachable:", box_up())

## 1 · Daten-Generierung (Isaac SDG) — generisch
Pro Teil: physik-basierte Drops, Top-Down-Render → `faceset_<part>/` mit RGB + Tiefe + Maske + Pose. Läuft auf der GPU-Box (`sim_code/render_dataset.py`). **Generisch**: jedes USD-Teil → Dataset, keine Per-Teil-Logik.

> Roadmap: **1.2** generischer Multi-Part-Renderer · **1.3** Domain-Randomization · **1.4** Sampling/Manifest.

In [ ]:
def generate_dataset(part, num=160):
    """Render a top-down drop dataset for one part on the GPU box."""
    assert wake_box(), "GPU box not reachable"
    push("sim_code")
    cmd = (f"cd {BOX_REPO} && {BOX_ISAAC} -u sim_code/render_dataset.py "
           f"--part {BOX_REPO}/{USD_DIR}/{part}.usdz "  # .usd fallback unten
           f"--out {BOX_REPO}/data/output/faceset_{part} --num {num}")
    print(on_box(cmd)[-600:])

# generate_dataset("Zahnrad_Typ7", num=160)   # auskommentiert — GPU-Lauf (~5 min)
print("→ generate_dataset(part) rendert ein Drop-Dataset (GPU). Bereits gerendert:", PARTS)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def show_dataset_samples(part, k=8):
    "Visualisiere ein paar gerenderte Top-Down-Crops (von der Box ziehen falls nötig)."
    d = OUT / f"faceset_{part}"
    if not d.exists():
        pull(f"data/output/faceset_{part}", f"data/output/faceset_{part}")
    rgbs = sorted(d.glob("rgb_*.png"))[:k]
    if not rgbs:
        print("kein Dataset lokal für", part, "— erst generate_dataset() oder pull()"); return
    fig, ax = plt.subplots(1, len(rgbs), figsize=(2*len(rgbs), 2.4))
    for a, p in zip(np.atleast_1d(ax), rgbs):
        a.imshow(Image.open(p)); a.set_axis_off()
    fig.suptitle(f"SDG-Drops — {part}"); plt.show()

# show_dataset_samples("Zahnrad_Typ7")

## 2 · Face-Atlas + Registry (pro Teil)
Aus den Drops die **distinkten Top-Ansichten (Faces)** clustern → `registry/<part>/faces_<part>.json` (R_face pro Face) + Templates. Das ist die „gelernte" Referenz, gegen die später klassifiziert + ausgerichtet wird. (`faces/cluster_views.py`)

> Generalisiert: g_body-Cluster (Orientierung bis auf Yaw) + Rotations-Match-Merge. Roadmap **2.1–2.3**.

In [ ]:
def build_registry(part):
    "Cluster faces + write registry on the box, then pull registry/<part> down."
    assert wake_box(), "GPU box not reachable"
    push("faces")
    on_box(f"cd {BOX_REPO}/faces && {BOX_FACES} cluster_views.py "
           f"{BOX_REPO}/data/output/faceset_{part} --part-name {part}")
    pull(f"data/output/faceset_{part}/faces", f"registry/{part}")
    print("registry/", part, "→", sorted((REGISTRY/part).glob("*")) if (REGISTRY/part).exists() else "—")

def show_face_atlas(part):
    reg = REGISTRY / part / f"faces_{part}.json"
    if not reg.exists(): print("keine Registry für", part); return
    faces = json.load(open(reg))["faces"]
    fig, ax = plt.subplots(1, max(2,len(faces)), figsize=(2.6*max(2,len(faces)), 3))
    for a, f in zip(np.atleast_1d(ax), faces):
        tp = REGISTRY/part/f"tmpl_{f['name'].replace(' ','')}.png"
        if tp.exists(): a.imshow(Image.open(tp))
        a.set_title(f"{f['name']}\n{f['prob']*100:.0f}%"); a.set_axis_off()
    fig.suptitle(f"Face-Atlas — {part}"); plt.show()

for p in PARTS: show_face_atlas(p)

## 3 · Model-Training (GETRENNT) — vorbereitet
Zwei getrennte Modelle:
1. **OBB-Detektor** — findet Teile + zieht oriented bounding boxes (trainiert auf SDG-OBB-Labels). *Roadmap 3.1 — noch zu bauen.*
2. **Face-Classifier pro Teil** — Crop → Face-ID (`faces/classifier/train.py`). *Roadmap 3.2 — Scaffold steht, Training offen.*

Bis trainiert: `infer()` nutzt **nearest-template-Fallback** (kein Checkpoint nötig) → die ganze Pipeline läuft bereits.

In [ ]:
def train_face_classifier(part):
    "Bereitet/startet das Face-CNN-Training (GPU/torch). NICHT in dieser Session ausgeführt."
    print(f"→ auf GPU-Box:  {BOX_FACES} faces/classifier/train.py --part {part}")
    print("   schreibt Checkpoint nach faces/classifier/checkpoints/<model>.pt")
    print("   infer() schaltet automatisch Fallback → CNN sobald der Checkpoint existiert.")

def train_obb_detector():
    raise NotImplementedError("Roadmap 3.1: OBB-Detektor-Training noch zu bauen "
                              "(YOLO-OBB o.ä. auf SDG bbox_2d/OBB-Labels).")

from faces.classifier import infer as face_infer
for p in PARTS:
    try: print(f"{p:24s} infer-backend: {face_infer.backend(p)}")
    except Exception as e: print(p, "infer-backend:", e)

## 4-5 · Inferenz: Bild → Teile → OBB → Crop → Face
Input = ein Bild + Detektionen (`bbox_2d` + Labels). Aktuell kommen die BBoxes als **Dummy aus der Simulation** (Platzhalter für den noch zu trainierenden OBB-Detektor, Roadmap 4.1). Pro Box: Crop → richtiges Face-Modell (per Label) → Face-ID. (`pipeline/inference.py`)

In [ ]:
from pipeline import inference as pinf

DUMMY_SCENE = REPO / "data" / "examples" / "dummy_scene"
def run_inference(scene_dir=DUMMY_SCENE):
    dets = pinf.run(str(scene_dir))   # [{instance_id, part, face, confidence, bbox_2d, ...}]
    print(f"{len(dets)} Detektionen")
    for d in dets[:6]:
        print(f"  #{d['instance_id']:>2} {d['part']:<22} {d.get('face','?'):<8} conf={d.get('confidence',0):.2f}")
    return dets

# dets = run_inference()   # entkommentieren wenn pipeline.inference.run existiert (siehe pipeline/)

## 6 · Alignment → volle 6D-Pose → `pose_result.json`
Pro Detektion: Seite (oben/unten) aus der **Face**, axialer **Yaw** indem die Face-Maske/Template iterativ in-plane gedreht + über den Crop gelegt wird (**MSE-Minimum**), `R_world = Rz(yaw)·R_face`, `t_world` per (x,y)-Backprojection auf die Tischebene. Ergebnis validiert gegen den Contract `docs/pose_result.schema.json`. (`pipeline/run_pipeline.py`)

In [ ]:
def run_full_pipeline(scene_dir=DUMMY_SCENE):
    "End-to-end: Bild+BBoxes → pose_result.json (Inferenz + Alignment)."
    r = subprocess.run(["bash","scripts/run_e2e.sh"], cwd=REPO, capture_output=True, text=True)
    print(r.stdout[-900:])
    pr = OUT / "pose_result.json"
    return json.load(open(pr)) if pr.exists() else None

result = run_full_pipeline()
if result:
    print("\nmeta:", result["meta"].get("schema_version"), "| Teile:", len(result["results"]))

## 7 · 3D-Anlage-Rekonstruktion
`pose_result.json` → Teile an ihrer 6D-Pose auf dem Tisch, relativ zum **Nullpunkt**. Inline eine schnelle 3D-Vorschau (matplotlib); der volle interaktive Viewer (Orbit, Klick-Info) ist `viewer/` (Three.js).

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa
def show_3d(result):
    if not result: print("kein result"); return
    fig = plt.figure(figsize=(7,6)); ax = fig.add_subplot(111, projection="3d")
    org = result["meta"].get("table_origin",[0,0,0])
    # Tischebene
    ax.plot([org[0]-.3,org[0]+.3,org[0]+.3,org[0]-.3,org[0]-.3],
            [org[1]-.3,org[1]-.3,org[1]+.3,org[1]+.3,org[1]-.3],
            [org[2]]*5, color="0.6")
    ax.scatter([org[0]],[org[1]],[org[2]], c="red", s=60, marker="x", label="Nullpunkt")
    for d in result["results"]:
        t = d["t_world"]; ax.scatter([t[0]],[t[1]],[t[2]], s=40)
        ax.text(t[0],t[1],t[2], f"{d['part']}\n{d.get('face','')}", fontsize=6)
    ax.set_title("3D-Anlage (Inline-Vorschau)"); ax.legend(); plt.show()
    print("Interaktiv (Orbit/Klick):  cd", REPO, "&& python3 -m http.server 8000")
    print("  → http://127.0.0.1:8000/viewer/?file=../data/output/pose_result.json")

show_3d(result)

## 8 · End-to-End: beliebiges CAD rein → 3D raus
Die generische Kette als ein Aufruf. Für ein **neues** Teil: USD nach `data/SDG/IsaacSim/USD-Files/` legen, in `PARTS` eintragen, dann:
```python
part = "MeinNeuesTeil"
generate_dataset(part)        # 1 · Drops rendern (GPU)
build_registry(part)          # 2 · Faces clustern → registry/
train_face_classifier(part)   # 3 · (optional) CNN trainieren; sonst Fallback
# 4-6 laufen über run_full_pipeline() sobald Detektor/Szene das Teil liefert
```
Sobald die Modelle trainiert + eingesteckt sind (`faces/classifier/checkpoints/`), läuft dieselbe Kette ohne Code-Änderung mit echten Confidences.

In [ ]:
print("Pipeline-Module:")
for m in ["sim_code/render_dataset.py (1 Daten)","faces/cluster_views.py (2 Atlas)",
          "faces/classifier/ (3 Training+infer)","pipeline/ (5-6 Inferenz+Alignment)",
          "viewer/ (7 3D)","scripts/run_e2e.sh (E2E)"]:
    print("  •", m)
print("\nRoadmap:  gantt status   |   Board:  kanban status")